In [ ]:
# ==============================================
# RETAIL ANALYTICS PIPELINE - COMPLETO
# ==============================================

# ----------------------------------------------
# 1. CONFIGURACIÓN DE SPARK
# ----------------------------------------------
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailAnalyticsPipeline") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

# ----------------------------------------------
# 2. DATOS SIMULADOS
# ----------------------------------------------
transactions = [
    (1, "user1", "productA", "electronics", 100.0),
    (2, "user2", "productB", "clothing", 50.0),
    (3, "user1", "productC", "electronics", 200.0),
    (4, "user3", "productD", "home", 80.0),
    (5, "user2", "productA", "electronics", 120.0),
    (6, "user4", "productE", "clothing", 60.0),
    (7, "user5", "productF", "home", 150.0),
    (8, "user6", "productG", "electronics", 300.0)
]

# ----------------------------------------------
# 3. RDDs
# ----------------------------------------------
rdd = sc.parallelize(transactions)

print("Total registros:", rdd.count())
print("Primeros registros:", rdd.take(3))

# Transformaciones
amounts = rdd.map(lambda x: x[4])
print("Suma total:", amounts.sum())

pair_rdd = rdd.map(lambda x: (x[3], x[4]))
revenue_by_category = pair_rdd.reduceByKey(lambda a, b: a + b)
print("Revenue por categoría:", revenue_by_category.collect())

# ----------------------------------------------
# 4. DATAFRAMES + SQL
# ----------------------------------------------
from pyspark.sql.types import *

schema = StructType([
    StructField("transaction_id", IntegerType(), True),
    StructField("user_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", FloatType(), True)
])

df = spark.createDataFrame(transactions, schema)

df.show()

# SQL
df.createOrReplaceTempView("transactions")

sales_by_category = spark.sql("""
    SELECT category, SUM(amount) as total_sales
    FROM transactions
    GROUP BY category
    ORDER BY total_sales DESC
""")

sales_by_category.show()

# Guardar Parquet
sales_by_category.write.mode("overwrite").parquet("sales_by_category.parquet")

# ----------------------------------------------
# 5. MACHINE LEARNING (MLlib)
# ----------------------------------------------
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Indexar
indexer = StringIndexer(inputCol="category", outputCol="category_index")
df_indexed = indexer.fit(df).transform(df)

# Features
assembler = VectorAssembler(
    inputCols=["amount", "category_index"],
    outputCol="features"
)

df_features = assembler.transform(df_indexed)

# Label (ejemplo)
df_final = df_features.withColumn("label", (df_features.amount > 100).cast("int"))

# Train/Test
train, test = df_final.randomSplit([0.7, 0.3])

# Modelo supervisado
lr = LogisticRegression(featuresCol="features", labelCol="label")
model = lr.fit(train)

predictions = model.transform(test)
predictions.select("features", "label", "prediction").show()

# Evaluación
evaluator = BinaryClassificationEvaluator()
auc = evaluator.evaluate(predictions)
print("AUC:", auc)

# KMeans
kmeans = KMeans(k=2, featuresCol="features")
kmeans_model = kmeans.fit(df_features)
clusters = kmeans_model.transform(df_features)

clusters.show()

# ----------------------------------------------
# 6. GRÁFICAS AUTOMÁTICAS
# ----------------------------------------------
import matplotlib.pyplot as plt

# --- Ventas por categoría ---
pdf_sales = sales_by_category.toPandas()

plt.figure()
plt.bar(pdf_sales['category'], pdf_sales['total_sales'])
plt.title("Ventas por Categoría")
plt.xlabel("Categoría")
plt.ylabel("Ventas")
plt.xticks(rotation=45)
plt.savefig("ventas_por_categoria.png")
plt.show()

# --- Top productos ---
from pyspark.sql.functions import sum as _sum

top_products = df.groupBy("product") \
    .agg(_sum("amount").alias("total_sales")) \
    .orderBy("total_sales", ascending=False)

pdf_products = top_products.toPandas()

plt.figure()
plt.bar(pdf_products['product'], pdf_products['total_sales'])
plt.title("Top Productos")
plt.xlabel("Producto")
plt.ylabel("Ventas")
plt.xticks(rotation=45)
plt.savefig("top_productos.png")
plt.show()

# --- Distribución ---
pdf_df = df.toPandas()

plt.figure()
plt.hist(pdf_df['amount'], bins=10)
plt.title("Distribución de Compras")
plt.xlabel("Monto")
plt.ylabel("Frecuencia")
plt.savefig("distribucion_compras.png")
plt.show()

# --- Clusters ---
pdf_clusters = clusters.toPandas()

plt.figure()
plt.scatter(pdf_clusters['amount'], pdf_clusters['category_index'])
plt.title("Clusters de Usuarios")
plt.xlabel("Monto")
plt.ylabel("Categoría Indexada")
plt.savefig("clusters.png")
plt.show()

# ----------------------------------------------
# 7. INSIGHTS
# ----------------------------------------------
print("\nINSIGHTS:")
print("- Se identificó la categoría con mayor ventas")
print("- Se segmentaron usuarios con KMeans")
print("- Modelo predice compras mayores a 100")

# ----------------------------------------------
# FIN
# ----------------------------------------------
